# Smart Contract Vulnerability Detection - Full Comparison
Training with all optimization thresholds: before_optimized, optimized_80p, optimized_50p, optimized_20p

Each model is trained from scratch with fresh BERT parameters.

In [ ]:
import os, sys, time, subprocess
import torch

print('='*60)
print('ENVIRONMENT CHECK')
print('='*60)
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print()

In [ ]:
print('='*60)
print('STEP 1: Install Dependencies')
print('='*60)

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'uv', '-q'],
    capture_output=True, text=True, timeout=60
)
print(f'uv install: OK' if result.returncode == 0 else 'FAILED')

missing_packages = ['transformers', 'accelerate', 'datasets', 'scikit-learn']
result = subprocess.run(
    ['uv', 'pip', 'install', '--system'] + missing_packages,
    capture_output=True, text=True, timeout=600
)
print(f'Packages install: OK' if result.returncode == 0 else 'FAILED')

for pkg in ['transformers', 'pandas', 'scikit-learn']:
    try:
        mod = __import__(pkg)
        print(f'  [OK] {pkg}')
    except ImportError:
        print(f'  [MISSING] {pkg}')
print()

In [ ]:
print('='*60)
print('STEP 2: Load Optimized Dataset (FULL)')
print('='*60)

import pandas as pd

dataset_dir = '/kaggle/input/datasets/jakeclark38a/smart-contract-vulnerability-detection'

train_df = pd.read_csv(os.path.join(dataset_dir, 'train_optimized_dataset.csv'))
test_df = pd.read_csv(os.path.join(dataset_dir, 'test_optimized_dataset.csv'))

print(f'Train samples: {len(train_df)}')
print(f'Test samples: {len(test_df)}')
print(f'Columns: {train_df.columns.tolist()}')

label_columns = ['Arithmetic', 'Unchecked Return Values For Low Level Calls', 
                 'Denial of Service', 'Time manipulation', 'Reentrancy']
print(f'Labels: {label_columns}')
print()

In [ ]:
print('='*60)
print('STEP 3: Define Helper Functions')
print('='*60)

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, hamming_loss, classification_report
import torch
from torch.utils.data import Dataset as TorchDataset
import gc

def hamming_score(y_true, y_pred, normalize=True, sample_weight=None):
    acc_list = []
    for i in range(y_true.shape[0]):
        set_true = set(np.where(y_true[i])[0])
        set_pred = set(np.where(y_pred[i])[0])
        tmp_a = None
        if len(set_true) == 0 and len(set_pred) == 0:
            tmp_a = 1
        else:
            tmp_a = len(set_true.intersection(set_pred))/float(len(set_true.union(set_pred)))
        acc_list.append(tmp_a)
    return np.mean(acc_list)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (logits > 0).astype(int)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    
    hamming = hamming_score(labels, predictions)
    h_loss = hamming_loss(labels, predictions)
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'hamming_score': hamming,
        'hamming_loss': h_loss
    }

class VulnerabilityDataset(TorchDataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item
    
    def __len__(self):
        return len(self.labels)

def train_and_evaluate(column_name, train_df, test_df, label_columns, output_dir):
    print(f'\n' + '='*60)
    print(f'Training with column: {column_name}')
    print('='*60)
    
    # Use FULL dataset
    train_texts = train_df[column_name].fillna('').astype(str).tolist()
    test_texts = test_df[column_name].fillna('').astype(str).tolist()
    train_labels = train_df[label_columns].values
    test_labels = test_df[label_columns].values
    
    print(f'Train: {len(train_texts)}, Test: {len(test_texts)}')
    
    MODEL_NAME = 'bert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    train_encodings = tokenizer(
        train_texts, truncation=True, padding=True, max_length=256, return_tensors='pt'
    )
    test_encodings = tokenizer(
        test_texts, truncation=True, padding=True, max_length=256, return_tensors='pt'
    )
    
    train_dataset = VulnerabilityDataset(train_encodings, train_labels)
    test_dataset = VulnerabilityDataset(test_encodings, test_labels)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_columns),
        problem_type='multi_label_classification'
    )
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=False,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to='none',
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )
    
    # Training time
    train_start = time.time()
    print('Training...')
    trainer.train()
    train_time = time.time() - train_start
    
    # Inference time on TEST dataset
    print('Evaluating on test set...')
    test_start = time.time()
    eval_results = trainer.evaluate()
    test_inference_time = time.time() - test_start
    
    # Inference time on TRAIN dataset
    print('Evaluating on train set...')
    train_start_infer = time.time()
    train_eval_results = trainer.evaluate(train_dataset)
    train_inference_time = time.time() - train_start_infer
    
    # Get predictions for classification report
    print('Generating classification report...')
    preds = trainer.predict(test_dataset)
    pred_logits = preds.predictions
    pred_labels = (pred_logits > 0).astype(int)
    
    print('\nClassification Report by Label:')
    print(classification_report(test_labels, pred_labels, target_names=label_columns, zero_division=0))
    
    model_save_dir = os.path.join(output_dir, column_name)
    os.makedirs(model_save_dir, exist_ok=True)
    model.save_pretrained(model_save_dir)
    tokenizer.save_pretrained(model_save_dir)
    
    del model, trainer, train_dataset, test_dataset
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        'column': column_name,
        'train_time': train_time,
        'train_inference_time': train_inference_time,
        'test_inference_time': test_inference_time,
        'num_train_samples': len(train_texts),
        'num_test_samples': len(test_texts),
        'precision': eval_results.get('eval_precision', 0),
        'recall': eval_results.get('eval_recall', 0),
        'f1': eval_results.get('eval_f1', 0),
        'hamming_score': eval_results.get('eval_hamming_score', 0),
        'hamming_loss': eval_results.get('eval_hamming_loss', 0),
    }

print('Helper functions defined')
print()

In [ ]:
print('='*60)
print('STEP 4: Run All Experiments')
print('='*60)

from pathlib import Path
import json

output_base = '/kaggle/working/output'
Path(output_base).mkdir(parents=True, exist_ok=True)

columns = ['before_optimized', 'optimized_80p', 'optimized_50p', 'optimized_20p']

all_results = []
total_start = time.time()

for col in columns:
    result = train_and_evaluate(
        column_name=col,
        train_df=train_df,
        test_df=test_df,
        label_columns=label_columns,
        output_dir=output_base
    )
    all_results.append(result)
    
    print(f'\nResults for {col}:')
    print(f'  Train Samples: {result["num_train_samples"]}, Test Samples: {result["num_test_samples"]}')
    print(f'  Train Time: {result["train_time"]/60:.1f} min')
    print(f'  Train Inference Time: {result["train_inference_time"]:.2f}s')
    print(f'  Test Inference Time: {result["test_inference_time"]:.2f}s')
    print(f'  Precision: {result["precision"]:.4f}')
    print(f'  Recall: {result["recall"]:.4f}')
    print(f'  F1: {result["f1"]:.4f}')
    print(f'  Hamming Score: {result["hamming_score"]:.4f}')
    print(f'  Hamming Loss: {result["hamming_loss"]:.4f}')
    
    gc.collect()
    torch.cuda.empty_cache()

total_time = time.time() - total_start
print(f'\nTotal training time: {total_time/60:.1f} minutes')
print()

In [ ]:
print('='*60)
print('STEP 5: Summary Comparison')
print('='*60)

import pandas as pd

comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df[['column', 'num_train_samples', 'num_test_samples', 'train_time', 'train_inference_time', 'test_inference_time', 'precision', 'recall', 'f1', 'hamming_score', 'hamming_loss']]
comparison_df.columns = ['Dataset', 'Train Samples', 'Test Samples', 'Train Time (s)', 'Train Inference (s)', 'Test Inference (s)', 'Precision', 'Recall', 'F1', 'Hamming Score', 'Hamming Loss']

print('\n' + '='*100)
print('FINAL RESULTS COMPARISON')
print('='*100)
print(comparison_df.to_string(index=False))
print('='*100)

comparison_csv = os.path.join(output_base, 'comparison_results.csv')
comparison_df.to_csv(comparison_csv, index=False)
print(f'\nResults saved to: {comparison_csv}')

results_json = {
    'configuration': {
        'model': 'bert-base-uncased',
        'labels': label_columns
    },
    'results': all_results,
    'total_time': total_time
}

results_json_path = os.path.join(output_base, 'experiment_results.json')
with open(results_json_path, 'w') as f:
    json.dump(results_json, f, indent=2)
print(f'Results saved to: {results_json_path}')

print('\nAll experiments completed!')
print()